# Kom Bish training

Settings → Accelerator **GPU T4** (use one GPU), **Internet On**, **Add Input → kombish-rice-raw**. Run all. Then **Save Version → Save & Run All (Commit)** so outputs persist. Outputs: `/kaggle/working/export/`.

In [ ]:
# Edit these two lines.
REPO_URL = "https://github.com/Raiyan465-F1/kom-bish.git"   # the repo must be public for this clone to work
RAW_ROOT = "/kaggle/input/kombish-rice-raw"    # real path from step 02
DATA, OUT = "/kaggle/working/data", "/kaggle/working/export"

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__)

In [ ]:
# onnxruntime 1.31 needs Python >= 3.11, so pick the version by the runtime. torch is NOT reinstalled.
import subprocess, sys
pkgs = ["timm==1.0.30", "onnx==1.23.2", "onnxscript==0.7.2", "datasets==5.1.0", "matplotlib"]
pkgs.append("onnxruntime==1.31.0" if sys.version_info >= (3, 11) else "onnxruntime")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
try:
    import sklearn
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)
import torch, timm, onnxruntime
print("torch", torch.__version__, "timm", timm.__version__, "ort", onnxruntime.__version__, "cuda", torch.cuda.is_available())

In [ ]:
!rm -rf /kaggle/working/kom-bish && git clone --depth 1 {REPO_URL} /kaggle/working/kom-bish
%cd /kaggle/working/kom-bish
!git log --oneline -1

In [ ]:
# Kept for the record: the raw folder tree and image counts.
!python training/prepare_data.py --print-tree --root {RAW_ROOT}

In [ ]:
# Add --accept-count-mismatch only if step 07 documented every difference in docs/gates.md.
!python training/prepare_data.py --root {RAW_ROOT} --out {DATA} --dhs

In [ ]:
!python training/train_export.py --data {DATA} --out {OUT} --device cuda

In [ ]:
import json, os
m = json.load(open(f"{OUT}/metrics.json"))
t = m["test"]
print("model", m["model"], "| date", m["date"])
print("T", m["temperature"], "clamped", m["temperature_clamped"], "| t", m["threshold"], "| margin", m["margin"],
      "| target met", m["target_met"], "| spray-relevant precision ok", m["spray_relevant_precision_ok"])
print("test n", t["n"], "| macro-F1 (no abstention)", t["macro_f1_no_abstention"], "| coverage", t["coverage"],
      "| selective accuracy", t["selective_accuracy"])
print("precision on BLB answers", t["precision_blb_answers"], "| on blast answers", t["precision_blast_answers"])
print("Dhan-Shomadhan field blast", m["dhs_field_blast"])
print("ONNX parity", m["onnx_parity_max_abs_diff"], "| ONNX MB", m["onnx_mb"], "| train seconds", m["train_seconds"])
for name in sorted(os.listdir(OUT)):
    print(f"{os.path.getsize(os.path.join(OUT, name)):>12,d}  {name}")
for name in ["model.onnx", "labels.json", "metrics.json", "confusion.png", "risk_coverage.png", "manifest.csv", "best.pt"]:
    assert os.path.exists(os.path.join(OUT, name)), f"missing {name}"
assert not os.path.exists(os.path.join(OUT, "model.onnx.data")), "model.onnx must be a single file"
print("All export files present.")

Download from the **Output** tab (`export/`), or from a laptop with `kagglehub.notebook_output_download('<kaggle-user>/<notebook-slug>')`. Then do step 10's hand-back.